# 🎵 歌词识别助手

基于 LangChain Agent + Tavily 搜索实现的智能歌词识别系统。输入一段歌词，Agent 自动调用搜索引擎查找对应歌曲、歌手及专辑信息，并以结构化格式输出。

**核心技术栈：**
- **LangChain**: Agent 框架 (`create_agent`)
- **DeepSeek**: 大语言模型（兼容 OpenAI 接口）
- **Tavily**: 网络搜索工具
- **Streaming**: 实时流式输出

## 1. 配置环境与依赖

In [20]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

load_dotenv()

base_url = os.getenv("DEEPSEEK_BASE_URL")
api_key = os.getenv("DEEPSEEK_API_KEY")

print(base_url)
print(api_key)

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
)

web_search = TavilySearch(
    max_results=3,
    topic="general",
    search_depth="basic",
)

https://api.deepseek.com
sk-74a02e92ad8a4140858fd0ac2160092b


## 2. 定义 System Prompt

系统提示词是 Agent 的核心指令，定义了歌词识别的完整工作流：搜索 → 补充检索 → 结构化输出。

In [21]:
system_prompt = """你是一个专业的歌词识别助手。收到用户提供的歌词后，请严格按以下流程处理：

步骤1 — 歌词检索
调用 web_search 工具搜索用户输入的歌词，定位歌曲、专辑和歌手信息。

步骤2 — 结构化输出
整合所有检索结果，按以下格式输出：

### 歌曲信息
- **歌曲名**：《歌曲名称》
- **对应专辑**：《专辑名》（发行年份）
- **作词/作曲**：如可查得

### 歌手信息
- **姓名**：
- **出生日期**：
- **出生地**：
- **主要成就**：
- **代表作品**：

### 数据来源
- [标题](链接)
- [标题](链接)

## 规则
- 必须调用 web_search 工具获取信息，禁止凭记忆回答
- 信息不足时标注"未公开"或"不详"
- 禁止编造任何细节，所有信息必须有可追溯来源
- 输出末尾列出所有参考链接
"""

## 3. 构造用户消息

将待识别的歌词封装为 `HumanMessage`，支持多段文本组合。

In [22]:
# 定义要识别的歌词
lyrics = "夜里拜山头"

human_message = HumanMessage(
    content=[
        {"type": "text", "text": "请帮我识别以下歌词"},
        {"type": "text", "text": lyrics},
    ]
)

## 4. 创建 Agent

使用 `create_agent` 将模型、工具和系统提示词组合成一个可自主决策的智能体。

In [23]:
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
)

## 5. 流式调用 Agent

通过 `stream()` 方法实现逐字输出，提升用户体验。包含异常处理以确保健壮性。

In [24]:
resp = agent.invoke({"messages": [human_message]})

In [25]:
from rich import print as rprint
rprint(resp)

{
    'messages': [
        HumanMessage(
            content=[{'type': 'text', 'text': '请帮我识别以下歌词'}, {'type': 'text', 'text': '夜里拜山头'}],
            additional_kwargs={},
            response_metadata={},
            id='82764065-b3c6-49b2-bb53-bff99dc49c5c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 2028,
                    'total_tokens': 2116,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 21,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1920},
                    'prompt_cache_hit_tokens': 1920,
                    'prompt_cache_miss_tokens': 108
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-v4-pro',
                'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
                'id': '025731c2-85c3-4e7b-955c-5daf072f56a9',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1e76-6280-7d73-add4-1389e26b9258-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {'query': '夜里拜山头 歌词', 'search_depth': 'advanced'},
                    'id': 'call_00_6yEe3cOuxBNigZpAdhsL5439',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2028,
                'output_tokens': 88,
                'total_tokens': 2116,
                'input_token_details': {'cache_read': 1920},
                'output_token_details': {'reasoning': 21}
            }
        ),
        ToolMessage(
            content='{"query": "夜里拜山头 歌词", "follow_up_questions": null, "answer": null, "images": [], 
"results": [{"url": "https://www.shazam.com/zh-tw/song/1726898542/%E5%9D%90%E5%BF%98%E9%81%93?tab=lyrics", "title":
"坐忘道- 王朝1982：歌詞、音樂影片和演唱會 - Shazam", "content": "下載 Shazam\\n\\n 取得此 App\\n 演唱會\\n 
排行榜\\n Radio Spins\\n Fast Forward \'26\\n\\n 說明\\n\\nalbum cover\\n\\n坐忘道 由 Global Savage Generation 
(GSG)於 2022年12月27日發行，收錄於專輯《坐忘道 - Single》中\\n\\nalbum cover\\n\\n專輯坐忘道 - 
Single\\n\\n發行日期2022年12月27日\\n\\n標籤Global Savage Generation 
(GSG)\\n\\n旋律\\n\\n歌曲有多麼清晰易記且符合明確音樂模式的旋律。通常，旋律分明的作品會擁有清晰易記的器樂或人聲主線
。\\n\\n原聲音質\\n\\n此指標衡量一首歌曲在多大程度上依賴原聲樂器 
(例如鋼琴、吉他、小提琴、鼓、薩克斯風)，而非電子或數位合成音效\\n\\nValence\\n\\n歌曲透過和聲與節奏所傳達的音樂積極
性或情感基調。數值高通常對應快樂、興奮或愉悅感，數值低則與悲傷、憤怒或憂鬱相關。\\n\\n節奏感\\n\\n綜合了節拍穩定性
、節奏型態與重拍強度等多重因素，以判定一首歌曲適合跳舞的程度。一首「節奏感強」的歌曲，通常具備穩定的速度、重複的音
樂結構與明顯的強拍。\\n\\n輕快\\n\\n曲目的律動感可能受節奏快慢、音量起伏與聲譜密度所影響。較輕快的歌曲通常節奏強勁
，編曲豐滿；反之，不太輕快的歌曲則可能編曲簡約、節奏較慢。\\n\\nBPM93\\n\\n## 概覽\\n\\n## 歌詞\\n\\n## 
歌詞\\n\\n[Intro]\\n\\n嘿嘿 只恐你来得 就去不得\\n\\n[Chorus]\\n\\n嘿 
夜里拜山头\\n\\n勾肩搭背是谁的手\\n\\n麻起胆子就跟到走\\n\\n一不做 二不休\\n\\n[Verse 1]\\n\\n嘿 
夜里拜山头\\n\\n胡言乱语都猜不透\\n\\n喜怒哀乐就揣心口\\n\\n忘情忘道忘春秋\\n\\n[Verse 2]\\n\\n天上飞鹞子 
地上跑豹子\\n\\n你娃干啥子 要上山发财\\n\\n有病莫乱投 坟地上乱游\\n\\n拜哪座山头 阴沟落进鞋子头\\n\\n[Verse 3] 
[...] 你娃干啥子 要上山发财\\n\\n有病莫乱投 坟地上乱游\\n\\n拜哪座山头 阴沟落进鞋子头\\n\\n[Verse 3]\\n\\n说得轻巧 
吃根灯草\\n\\n听到风就是雨 跟我 鬼扯 野扯\\n\\n几根堂 无事不登三宝殿\\n\\n穿的烂 个个都装金罗汉\\n\\n[Verse 
4]\\n\\n雾里看花花不在\\n\\n瞒天过海盗蓬莱\\n\\n鬼话哪有人心坏\\n\\n你别大惊小怪\\n\\n[Verse 
5]\\n\\n不染淤泥染尘埃\\n\\n都是肉眼看凡胎\\n\\n哪有真假和黑白\\n\\n我一粟压沧海\\n\\n[Chorus]\\n\\n嘿 
夜里拜山头\\n\\n勾肩搭背是谁的手\\n\\n麻起胆子就跟到走\\n\\n一不做 二不休\\n\\n[Chorus]\\n\\n嘿 
夜里拜山头\\n\\n胡言乱语都猜不透\\n\\n喜怒哀乐就揣心口\\n\\n忘情忘道忘春秋\\n\\n[Verse 6]\\n\\n龙凤旗 花花旗 
日月旗\\n\\n出来飘的人都想打个好字旗\\n\\n不怕老子有三张脸\\n\\n只怕有些人起两样心\\n\\n当面是人 
背后是鬼\\n\\n昌你祖坟 吐你口水\\n\\n天神不纳 地神不收\\n\\n夜路走多了 紧防被天收\\n\\n[Verse 
7]\\n\\n有财能使鬼退灾\\n\\n摇钱树下摘金牌\\n\\n叫得仙来仙要在\\n\\n唤得神去神不怪\\n\\n[Verse 
8]\\n\\n不染

In [27]:
for message in resp['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '请帮我识别以下歌词'}, {'type': 'text', 'text': '夜里拜山头'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_00_6yEe3cOuxBNigZpAdhsL5439)
 Call ID: call_00_6yEe3cOuxBNigZpAdhsL5439
  Args:
    query: 夜里拜山头 歌词
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "夜里拜山头 歌词", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.shazam.com/zh-tw/song/1726898542/%E5%9D%90%E5%BF%98%E9%81%93?tab=lyrics", "title": "坐忘道- 王朝1982：歌詞、音樂影片和演唱會 - Shazam", "content": "下載 Shazam\n\n 取得此 App\n 演唱會\n 排行榜\n Radio Spins\n Fast Forward '26\n\n 說明\n\nalbum cover\n\n坐忘道 由 Global Savage Generation (GSG)於 2022年12月27日發行，收錄於專輯《坐忘道 - Single》中\n\nalbum cover\n\n專輯坐忘道 - Single\n\n發行日期2022年12月27日\n\n標籤Global Savage Generation (GSG)\n

## 📝 小结

本 Notebook 展示了 LangChain Agent 的核心用法：

1. **Agent 创建** — `create_agent(model, tools, system_prompt)` 一行即可创建
2. **工具集成** — Tavily 搜索引擎作为 Agent 的"外脑"
3. **流式输出** — `stream()` 方法提供实时反馈
4. **结构化提示词** — 精心设计的 system prompt 是 Agent 质量的关键

**扩展建议：**
- 可替换为其他 LLM（如 GPT-4o、Claude 等）
- 可集成更多工具（如 Wikipedia、数据库查询）
- 可添加记忆组件实现多轮对话